# Transform Constructors Data

1. Read bronze `constructors` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`)
1. Rename columns to make them more meaningful (`name` → `constructor_name`)
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `constructors` table

In [0]:

dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")  

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
const_df = spark.read.table(bronze_table).filter((F.col("batch_id") == v_batch_id))

In [0]:
const_drop_url_df = const_df.drop("url")

In [0]:
const_renamed_col_df = const_drop_url_df.withColumnsRenamed(
    {"constructorId": "constructor_id", "name": "constructor_name"}
)

In [0]:
const_removed_duplicates = const_renamed_col_df.dropDuplicates()

In [0]:
const_final_df = const_removed_duplicates.withColumn(
    "nationality", F.initcap(F.col("nationality"))
)

In [0]:
write_to_silver(
    const_final_df,
    silver_table,
    "t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
%sql
select
  *
from
  formula1_incr.silver.constructors